In [1]:
!nvidia-smi

Sat Jan 31 19:25:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               Off |   00000000:AC:00.0  On |                  Off |
| 73%   81C    P0            206W /  300W |   11657MiB /  49140MiB |      7%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from unsloth import FastLanguageModel
import torch
from transformers import TextStreamer

model_name = "unsloth/Qwen3-32B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    max_seq_length=4096,
    device_map = "auto",
)

FastLanguageModel.for_inference(model)

custom_template = """{% for message in messages %}
{% if message['role'] == 'developer' or message['role'] == 'system' %}
<|im_start|>system
{{ message['content'] }}<|im_end|>
{% elif message['role'] == 'user' %}
<|im_start|>user
{{ message['content'] }}<|im_end|>
{% elif message['role'] == 'assistant' %}
{% if message['description'] %}
<|im_start|>assistant
<final>
{{ message['description'] }}
</final>
<|im_end|>
{% else %}
<|im_start|>assistant
<think>
{{ message['thinking'] }}</think>
<final>
{{ message['content'] }}
</final><|im_end|>
{% endif %}
{% endif %}
{% endfor %}
{%- if add_generation_prompt %}
{%- if think %}
<|im_start|>assistant
<think>
{%- else %}
<|im_start|>assistant
<final>
{%- endif %}
{%- endif %}"""

# Iniezione nel tokenizer
tokenizer.chat_template = custom_template


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-01-31 19:25:47.656418: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-31 19:25:47.666928: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769883947.679617 3709450 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769883947.683585 3709450 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769883947.694386 3709450 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Unsloth: Your Flash Attention 2 installation seems to be broken?
A possible explanation is you have a new CUDA version which isn't
yet compatible with FA2? Please file a ticket to Unsloth or FA2.
We shall now use Xformers instead, which does not have any performance hits!
We found this negligible impact by benchmarking on 1x A100.
Switching to PyTorch attention since your Xformers is broken.

/home/habes/anaconda3/envs/llmenv/lib/python3.10/site-packages/flash_attn_2_cuda.cpython-310-x86_64-linux-gnu.so: undefined symbol: _ZN3c105ErrorC2ENS_14SourceLocationESs
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.10.12: Fast Qwen3 patching. Transformers: 4.57.1.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 1. Max memory: 47.422 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unslo

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [3]:
def generate_response(messages, with_stream=True, think=True):
    # Ora apply_chat_template userà il tuo schema personalizzato
    inputs_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        return_tensors = "pt",
        think = think, # Questo parametro ora viene letto dal tuo template Jinja
        max_length = 4096,
    ).to(model.device)

    if with_stream:
        text_streamer = TextStreamer(tokenizer, skip_prompt = True)
    else:
        text_streamer = None

    out = model.generate(
        inputs_ids, 
        streamer = text_streamer, 
        max_new_tokens = 4096, 
        temperature=0.3, 
        top_p=0.90,
        # Importante: definisci il token di fine per evitare loop
        eos_token_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    )

    generated_tokens = out[0][len(inputs_ids[0]):]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)

In [4]:
!rm -r ./test_contracts/*/.ipynb_checkpoints
!rm -r ./test_contracts/.ipynb_checkpoints

!ls -la ./test_contracts

rm: cannot remove './test_contracts/*/.ipynb_checkpoints': No such file or directory


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


rm: cannot remove './test_contracts/.ipynb_checkpoints': No such file or directory
total 188
drwxr-xr-x  2 habes habes 4096 Nov 25 20:31 .
drwxrwxr-x 13 habes habes 4096 Jan 30 18:36 ..
-rw-r--r--  1 habes habes 1487 Nov 18 17:00 pyteal10.json
-rw-r--r--  1 habes habes 2155 Nov 18 17:00 pyteal11.json
-rw-r--r--  1 habes habes 2236 Nov 18 17:00 pyteal12.json
-rw-r--r--  1 habes habes  882 Nov 18 17:00 pyteal13.json
-rw-r--r--  1 habes habes 2750 Nov 18 17:00 pyteal14.json
-rw-r--r--  1 habes habes 1829 Nov 18 17:00 pyteal15.json
-rw-r--r--  1 habes habes 1044 Nov 18 17:00 pyteal16.json
-rw-r--r--  1 habes habes 2479 Nov 18 17:00 pyteal17.json
-rw-r--r--  1 habes habes 2467 Nov 18 17:00 pyteal18.json
-rw-r--r--  1 habes habes 2072 Nov 18 17:00 pyteal19.json
-rw-r--r--  1 habes habes 1436 Nov 18 17:00 pyteal1.json
-rw-r--r--  1 habes habes 1588 Nov 18 17:00 pyteal20.json
-rw-r--r--  1 habes habes 1967 Nov 18 17:00 pyteal21.json
-rw-r--r--  1 habes habes 2908 Nov 25 19:30 pyteal22.json
-rw

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [4]:
# RAG CHECKLIST

import os
import json
from test_logs import *
from rag import search_similarity, parse_response, map_vulnerability, create_rag_checklist
from prompts import dev_description_prompt, user_description_prompt

system_prompt = """
You are an expert smart contract security auditor specialized in the Algorand blockchain and PyTeal.  
Your task is to analyze PyTeal code precisely and systematically to identify security vulnerabilities, **leveraging additional contextual information retrieved via RAG** (i.e., probable vulnerabilities provided to you).

### Required behavior:
- Always perform a structured, explicit reasoning phase first and put it inside the <think> block.
  - In <think>...</think> you must:
    - Summarize the contract’s purpose and high-level architecture.
    - Inspect the logic block-by-block (or line-by-line for short snippets).
    - For each **RAG-suggested vulnerability**, check whether the code enforces the required security invariant; note any matches or gaps.
    - Also remain open to detecting **other vulnerabilities** not present in the RAG list.
    - Use concise, technical language and show the chain of reasoning (why you suspect an issue, how you trace it to specific code).

- After </think>, provide the final judgment in the <final>…</final> block using this exact structure:
  - A short summary sentence (one or two lines).
  - ### Vulnerability: <name or "None detected">
  - ### Explanation: <concise cause and how it could be exploited>
  - ### Risk: <severity (Critical / High / Medium / Low) and short impact statement>
  - ### Mitigation: <for each identified issue, a concrete fix or recommendation>

### Constraints:
- Do NOT include any extra commentary, greetings, or meta-text.
- If no vulnerability is found, explicitly write “### Vulnerability: Not Vulnerable” and “### Risk: No significant risk identified”.
- Do NOT consider logical flaws as vulnerabilities.
"""

user_prompt = """
Here is a **vulnerability checklist** (from RAG) with your priority guidance. Use it to steer your audit, but also be ready to discover new issues beyond this list.

{checklist}

Now perform a detailed security analysis of the following PyTeal smart contract using the reasoning structure.

Contract Code:

```python
{code}
```
"""

test_dir = "./test_contracts"

test_logs_basedir = "./test_logs_prova"

test_logs_dir = create_test_log_dir(test_logs_basedir, model_name)

for file in os.listdir(test_dir):

    full_path = os.path.join(test_dir,file)

    with open(full_path, "r", encoding="utf-8") as f:
            content = json.load(f)
            code = content['smart_contract']
            vulnerability = content['vulnerability']
    
    log_file = create_log_file(test_logs_dir, file, vulnerability)
    write_log_summary(test_logs_dir, f"{'-'*50}ANALYZING {file}-{vulnerability}{'-'*50}")
    
    # CREATE A DESCRIPTION OF THE CODE USING THE MODEL
    messages_description = [
            {"role": "developer", "content": dev_description_prompt},
            {"role": "user", "content": user_description_prompt.format(code=code)},
        ]
    
    description = generate_response(messages_description,think=False, with_stream=False)
    write_log(log_file, description, type="Description")
    
    # RETRIEVE A LIST OF THE MOST SIMILAR CONTRACT
    rag_contracts = search_similarity(description)

    checklist = create_rag_checklist(rag_contracts, with_few_shot=True)

    write_log(log_file, checklist, type="Checklist")

    messages = [
            {"role": "developer", "content": system_prompt},
            {"role": "user", "content": user_prompt.format(checklist=checklist, code=code)},
        ]
    
    # MODEL FIRST RESPONSE THAT CONTAIN POTENTIAL VULNERABILITIES
    response = generate_response(messages, with_stream=False)
    write_log(log_file, response, type="Audit")
    
    # PARSE THE RESPONSE TO EXTRACT A LIST OF VULNERABILITIES
    vulnerabilities = parse_response(response)
    write_log_summary(test_logs_dir, f"Vulnerabilities found in the audit: {vulnerabilities}")

Loading embedding model: hkunlp/instructor-xl


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


--------------------------------------------------ANALYZING pyteal43.json-Unchecked_Payment_Receiver--------------------------------------------------
Vulnerabilities found in the audit: ['Unchecked Payment Receiver']
--------------------------------------------------ANALYZING pyteal25.json-no vuln--------------------------------------------------
Vulnerabilities found in the audit: []
--------------------------------------------------ANALYZING pyteal18.json-close_remainder_to--------------------------------------------------
Vulnerabilities found in the audit: ['Unchecked Close Remainder To']
--------------------------------------------------ANALYZING pyteal17.json-close_remainder_to--------------------------------------------------
Vulnerabilities found in the audit: ['Not Vulnerable']
--------------------------------------------------ANALYZING pyteal19.json-close_remainder_to--------------------------------------------------
Vulnerabilities found in the audit: ['Unchecked Close Rema

In [4]:
#NO RAG

import os
import json
from test_logs import *
from rag import parse_response


test_dir = "./test_contracts"


system_prompt = """
You are an expert smart contract security auditor specialized in the Algorand blockchain and PyTeal. 
Your task is to analyze PyTeal code precisely and systematically to identify security vulnerabilities.

### Required behavior:
- Always perform a structured, explicit reasoning phase first and put it inside the <think> block.
  - In <think>...</think> you must:
    - Summarize the contract's purpose and high-level architecture.
    - Inspect the logic block-by-block (or line-by-line for short snippets).
    - Note any suspicious patterns, missing authorization checks, unsafe transaction handling, or other issues with evidence (point to the code lines/constructs).
    - Use concise, technical language and show the chain of reasoning (why you suspect an issue).

- After </think>, provide the final judgment in the <final>...</final> block using this exact structure:
  - A short summary sentence.
  - ### Vulnerability: <name or "Not Vulnerable">
  - ### Explanation: <concise cause and how it can be exploited>
  - ### Risk: <severity (Critical/High/Medium/Low) and short impact statement>
"""


user_prompt = """
Here is a vulnerability checklist derived from Retrieval-Augmented Generation (RAG), with priorities.
Use this checklist to guide your analysis, but DO NOT limit your audit to only the vulnerabilities listed.
All other potential vulnerabilities must still be considered.

Important rules:
- Each vulnerability MUST be evaluated independently.
- A vulnerability is present if its security checks are missing AND its preconditions apply.
- The absence of a check MUST be treated as unsafe unless the preconditions clearly do not apply.
- Do NOT assume safety from default values, implicit behavior, or omitted fields.
- If a vulnerability is not applicable, explicitly state why in your reasoning.

---

### Vulnerabilities to evaluate

#### 1. Unchecked Transaction Fee
- **Security check:** Ensure that ALL critical transactions explicitly enforce a maximum fee
  (`Txn.fee()` or `Gtxn[i].fee()`).
  Critical transactions include:
  - Payments
  - Asset Transfers
  - Application Calls (when authorized by a smart signature)
- **Preconditions:** Applies ONLY to smart signatures (delegated logic signatures or contract accounts)
  that authorize transactions where the sender is the victim of an excessive fee.

---

#### 2. Unchecked Close Remainder To
- **Security check:** Ensure that ALL authorized `TxnType.Payment` transactions explicitly validate:
  - `Txn.close_remainder_to() == Global.zero_address()`
  - OR restrict it to a known, trusted address if account closure is intentionally supported.
- **Important:** The absence of an explicit check does NOT imply safety.
- **Preconditions:** Applies ONLY to stateless smart contracts (logic signatures) that authorize payment transactions.

---

#### 3. Unchecked Asset Close To
- **Security check:** Ensure that ALL authorized `TxnType.AssetTransfer` transactions explicitly validate:
  - `Txn.asset_close_to() == Global.zero_address()`
  - OR restrict it to a predefined trusted address.
- **Preconditions:** Applies ONLY to smart signatures that authorize asset transfer transactions.

---

#### 4. Unchecked Rekey To
- **Security check:** Ensure that ALL authorized Payment and Asset Transfer transactions explicitly validate:
  - `Txn.rekey_to() == Global.zero_address()`
  - or `Gtxn[i].rekey_to() == Global.zero_address()`
- **Important:** Missing checks MUST be treated as vulnerabilities.
- **Preconditions:**
  - Applies ONLY to smart signatures
  - Applies ONLY to Payment and Asset Transfer transactions
  - ApplicationCall transactions cannot be rekeyed and are therefore excluded.

---

#### 5. Unchecked Payment Receiver
- **Security check:** Ensure that ALL authorized `TxnType.Payment` transactions explicitly restrict
  the receiver to whitelisted, trusted addresses.
- **Preconditions:** Applies ONLY to Payment transactions where the contract authorizes fund transfers
  (e.g., escrows, withdrawals, refunds).

---

#### 6. Unchecked Asset Receiver
- **Security check:** Ensure that ALL authorized `TxnType.AssetTransfer` transactions explicitly restrict
  the asset receiver to trusted addresses.
- **Preconditions:** Applies to contracts managing ASA distributions, vesting, redemptions,
  or similar asset-handling logic.

---

#### 7. Arbitrary Update
- **Security check:** Ensure that `OnComplete.UpdateApplication` is:
  - either not handled at all
  - OR explicitly restricted to an authorized entity (creator or admin).
- **Preconditions:** Applies ONLY to stateful smart contracts that handle `UpdateApplication`.

---

#### 8. Arbitrary Delete
- **Security check:** Ensure that `OnComplete.DeleteApplication` is:
  - either not handled at all
  - OR explicitly restricted to an authorized entity (creator or admin).
- **Preconditions:** Applies ONLY to stateful smart contracts that handle `DeleteApplication`.

---

### PyTeal Smart Contract Code

```pyteal
{code}


"""

test_logs_basedir = "./test_logs_no_rag"

test_logs_dir = create_test_log_dir(test_logs_basedir, model_name)

for file in os.listdir(test_dir):

    full_path = os.path.join(test_dir,file)

    with open(full_path, "r", encoding="utf-8") as f:
            content = json.load(f)
            code = content['smart_contract']
            vulnerability = content['vulnerability']
    
    log_file = create_log_file(test_logs_dir, file, vulnerability)
    write_log_summary(test_logs_dir, f"{'-'*50}ANALYZING {file}-{vulnerability}{'-'*50}")

    messages = [
            {"role": "developer", "content": system_prompt},
            {"role": "user", "content": user_prompt.format(code=code)},
        ]
    
    # MODEL FIRST RESPONSE THAT CONTAIN POTENTIAL VULNERABILITIES
    response = generate_response(messages, with_stream=False)
    write_log(log_file, response, type="Audit")
    
    # PARSE THE RESPONSE TO EXTRACT A LIST OF VULNERABILITIES
    vulnerabilities = parse_response(response)
    write_log_summary(test_logs_dir, f"Vulnerabilities found in the audit: {vulnerabilities}")

Loading embedding model: hkunlp/instructor-xl


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


--------------------------------------------------ANALYZING pyteal43.json-Unchecked_Payment_Receiver--------------------------------------------------
Vulnerabilities found in the audit: ['Arbitrary update']
--------------------------------------------------ANALYZING pyteal25.json-no vuln--------------------------------------------------
Vulnerabilities found in the audit: ['Not Vulnerable']
--------------------------------------------------ANALYZING pyteal18.json-close_remainder_to--------------------------------------------------
Vulnerabilities found in the audit: ['Unchecked Close Remainder To']
--------------------------------------------------ANALYZING pyteal17.json-close_remainder_to--------------------------------------------------
Vulnerabilities found in the audit: ['Unchecked Close Remainder To']
--------------------------------------------------ANALYZING pyteal19.json-close_remainder_to--------------------------------------------------
Vulnerabilities found in the audit: ['